In [ ]:
%sql
--------------------------------------------------------------------------------------------------------------
--- Epic Clarity Condition Era - Adapted to Databricks SQL from Pure SQL condition_era written by Chris_Knoll
--- https://gist.github.com/chrisknoll/c820cc12d833db2e3d1e
--- Uses omop_epic schema for Epic Clarity DQD testing
--- INTERVAL set to 30 days
--------------------------------------------------------------------------------------------------------------
TRUNCATE TABLE _exponent.omop_epic.condition_era;

WITH cteConditionTarget AS
(
	SELECT
		co.condition_occurrence_id,
		co.person_id,
		co.condition_concept_id,
		co.condition_start_date,
		COALESCE(co.condition_end_date, date_add(co.condition_start_date, 1)) AS condition_end_date
	FROM _exponent.omop_epic.condition_occurrence co
	WHERE condition_concept_id != 0
),
cteEndDates AS
(
	SELECT
		person_id,
		condition_concept_id,
		date_add(event_date, -30) AS end_date
	FROM
	(
		SELECT
			person_id,
			condition_concept_id,
			event_date,
			event_type,
			MAX(start_ordinal) OVER (PARTITION BY person_id, condition_concept_id ORDER BY event_date, event_type ROWS UNBOUNDED PRECEDING) AS start_ordinal,
			ROW_NUMBER() OVER (PARTITION BY person_id, condition_concept_id ORDER BY event_date, event_type) AS overall_ord
		FROM
		(
			SELECT
				person_id,
				condition_concept_id,
				condition_start_date AS event_date,
				-1 AS event_type,
				ROW_NUMBER() OVER (PARTITION BY person_id, condition_concept_id ORDER BY condition_start_date) AS start_ordinal
			FROM cteConditionTarget
			UNION ALL
			SELECT
				person_id,
				condition_concept_id,
				date_add(condition_end_date, 30) AS event_date,
				1 AS event_type,
				NULL AS start_ordinal
			FROM cteConditionTarget
		) RAWDATA
	) e
	WHERE (2 * e.start_ordinal) - e.overall_ord = 0
),
cteConditionEnds AS
(
SELECT
    c.person_id,
	c.condition_concept_id,
	c.condition_start_date,
	MIN(e.end_date) AS era_end_date
FROM cteConditionTarget c
JOIN cteEndDates e ON c.person_id = e.person_id AND c.condition_concept_id = e.condition_concept_id AND e.end_date >= c.condition_start_date
GROUP BY
    c.condition_occurrence_id,
	c.person_id,
	c.condition_concept_id,
	c.condition_start_date
)
INSERT INTO _exponent.omop_epic.condition_era (person_id, condition_concept_id, condition_era_start_date, condition_era_end_date, condition_occurrence_count)
SELECT
	person_id,
	condition_concept_id,
	MIN(condition_start_date) AS condition_era_start_date,
	era_end_date AS condition_era_end_date,
	COUNT(*) AS condition_occurrence_count
FROM cteConditionEnds
GROUP BY person_id, condition_concept_id, era_end_date
ORDER BY person_id, condition_concept_id
;